# Convert GeoJSON into PMTiles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kalimaha/wa-critical-infrastructure-map/blob/first-setup/notebooks/02_roads_transformation.ipynb)

In [1]:
import os
import subprocess

GEOJSON_FULL_TEMP = "../data/processed/wa_roads_full_temp.geojson"
PMTILES_OUTPUT = "../data/processed/Road_network.pmtiles"


tippecanoe_cmd = [
    "tippecanoe",
    "-o", PMTILES_OUTPUT,
    "-zg",
    "--drop-densest-as-needed",
    "--extend-zooms-if-still-dropping",
    "--simplify-only-low-zooms",
    "-y", "ROAD_NAME",
    "-y", "COMMON_USAGE_NAME",
    "-y", "NETWORK_TYPE",
    "--force",
    "../data/raw/Road_Network.geojson" # Passa direttamente il file raw
]

print("Generate PMTiles with Tippecanoe: START")
result = subprocess.run(tippecanoe_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Generate PMTiles with Tippecanoe: DONE - {PMTILES_OUTPUT}")
    if os.path.exists(GEOJSON_FULL_TEMP):
        os.remove(GEOJSON_FULL_TEMP)
else:
    print(f"Error:\n{result.stderr}")

Generate PMTiles with Tippecanoe: START


Generate PMTiles with Tippecanoe: DONE - ../data/processed/Road_network.pmtiles


In [2]:
import os
import subprocess

LGA_PMTILES_OUTPUT = "../data/processed/LGA_Boundaries.pmtiles"

tippecanoe_cmd = [
    "tippecanoe",
    "-o", LGA_PMTILES_OUTPUT,
    "-zg",
    "--drop-densest-as-needed",
    "--extend-zooms-if-still-dropping",
    "--simplify-only-low-zooms",
    "--no-tiny-polygon-reduction",
    "--detect-shared-borders",
    "--force",
    "../data/raw/LGA_Boundaries_LGATE_233_WA_GDA2020_Public.geojson",
]

print("Generate LGA boundary PMTiles with Tippecanoe: START")
result = subprocess.run(tippecanoe_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Generate LGA boundary PMTiles with Tippecanoe: DONE - {LGA_PMTILES_OUTPUT}")
else:
    print(f"Error:\n{result.stderr}")

Generate LGA boundary PMTiles with Tippecanoe: START


Generate LGA boundary PMTiles with Tippecanoe: DONE - ../data/processed/LGA_Boundaries.pmtiles


In [3]:
import glob
import os
import subprocess

import geopandas as gpd

SHAPE_DIR = "../data/raw/WAPoliceForceFacilities_SHP"
OUTPUT_DIR = "../data/processed"

for shp_path in sorted(glob.glob(os.path.join(SHAPE_DIR, "*.shp"))):
    base_name = os.path.splitext(os.path.basename(shp_path))[0]
    geojson_path = os.path.join(OUTPUT_DIR, f"{base_name}.geojson")
    pmtiles_path = os.path.join(OUTPUT_DIR, f"{base_name}.pmtiles")

    print(f"Convert {base_name} shapefile to GeoJSON: START")
    gdf = gpd.read_file(shp_path)
    gdf.to_file(geojson_path, driver="GeoJSON")
    print(f"Convert {base_name} shapefile to GeoJSON: DONE - {geojson_path}")

    tippecanoe_cmd = [
        "tippecanoe",
        "-o", pmtiles_path,
        "-zg",
        "--drop-densest-as-needed",
        "--extend-zooms-if-still-dropping",
        "--simplify-only-low-zooms",
        "--force",
        geojson_path,
    ]

    print(f"Generate PMTiles for {base_name} with Tippecanoe: START")
    result = subprocess.run(tippecanoe_cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"Generate PMTiles for {base_name}: DONE - {pmtiles_path}")
        if os.path.exists(geojson_path):
            os.remove(geojson_path)
    else:
        print(f"Error for {base_name}:\n{result.stderr}")
        raise RuntimeError(f"Tippecanoe failed for {shp_path}")


Convert Police_Facilities shapefile to GeoJSON: START


Convert Police_Facilities shapefile to GeoJSON: DONE - ../data/processed/Police_Facilities.geojson
Generate PMTiles for Police_Facilities with Tippecanoe: START
Generate PMTiles for Police_Facilities: DONE - ../data/processed/Police_Facilities.pmtiles


Start a local server with:

```
python3 -m http.server 8000
```